# Step 2: pre transfer success models

Question: for a player moving from one of nine European leagues outside the top five into a top five league, can we predict from data **before the move** whether he becomes a regular starter within two seasons?

Label `regular_starter`: in year 1 or year 2 after the move, the player played at least 50% of the possible league minutes of his new club (league games only, at the new club). Fixed before any model was run.

Run order: `python -m src.load_postgres`, then `psql -f sql/01_build_pre_transfer.sql`, then this notebook (`jupyter nbconvert --to notebook --execute notebooks/02_models.ipynb`).

In [1]:
import sys, json, datetime
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, psycopg
from sklearn.metrics import roc_auc_score
from src.modeling import *
RUN_DATE = datetime.date.today().isoformat()
print('run date', RUN_DATE)

run date 2026-10-15


## 1. How many real records? (the resume says 3500+)

In [2]:
with psycopg.connect(db_url()) as conn:
    funnel = pd.read_sql('SELECT step, description, n FROM tm.funnel ORDER BY step', conn)
funnel

,step,description,n
0,1,rows in transfers table,175165
1,2,dated on or before snapshot 2026-07-06,174645
2,3,dated 2013-06-01 to 2024-09-30 (label window c...,133276
3,4,from club in one of the 9 source leagues,13519
4,5,and to club in a top five league (GB1 ES1 IT1 ...,1725
5,6,after dropping same club moves and duplicate rows,1725
6,7,pre transfer records with at least 1 appearanc...,1669


In [3]:
df = load_records()
train, val, test, embargo = split(df)
splits = pd.DataFrame({'rows': [len(train), len(embargo), len(val), len(test)],
    'regular_starter_rate': [train.regular_starter.mean(), embargo.regular_starter.mean(), val.regular_starter.mean(), test.regular_starter.mean()],
    'transfer_dates': [f'{train.transfer_date.min().date()} to {train.transfer_date.max().date()}',
                       f'{embargo.transfer_date.min().date()} to {embargo.transfer_date.max().date()} (not used)',
                       f'{val.transfer_date.min().date()} to {val.transfer_date.max().date()}',
                       f'{test.transfer_date.min().date()} to {test.transfer_date.max().date()}']},
    index=['train', 'embargo', 'validation', 'test'])
print('Total pre transfer records:', len(df))
splits

Total pre transfer records: 1669


,rows,regular_starter_rate,transfer_dates
train,694,0.325648,2013-06-30 to 2020-06-30
embargo,396,0.219697,2020-07-01 to 2022-06-30 (not used)
validation,246,0.317073,2022-07-01 to 2023-06-30
test,333,0.309309,2023-07-01 to 2024-08-30


The embargo drops moves from 2020-07 to 2022-06: their two year label windows would run into the validation period, so training on them would let the model see outcomes from the future of the validation moves. It costs 396 records. Hyperparameters are chosen on validation ROC AUC only. The test set is scored once, below.

## 2. Fit all models

In [4]:
models = fit_all(train, val)
for k, v in models.items():
    print(f"{k:22s} params={v['params']}  val_auc={v.get('val_auc')}")

baseline_prior         params={'prior': 0.326}  val_auc=None
baseline_minutes       params={'rule': 'minutes_365 / 3400'}  val_auc=None
logistic_regression    params={'C': 0.01}  val_auc=0.7612179487179488
gradient_boosting      params={'learning_rate': 0.03, 'max_depth': 2, 'n_estimators': 100}  val_auc=0.7374847374847375
pytorch_mlp            params={'hidden': [32, 16], 'dropout': 0.4, 'weight_decay': 0.01, 'epochs_run': 47}  val_auc=0.7641178266178267


## 3. One results table (test set, threshold picked on validation by best F1)

In [5]:
ytr_val = val.regular_starter.values
yte = test.regular_starter.values
rows = []
for name, m in models.items():
    pv, pt = m['predict'](val), m['predict'](test)
    thr = best_f1_threshold(ytr_val, pv) if name != 'baseline_minutes' else best_f1_threshold(ytr_val, pv)
    r = metrics(yte, pt, thr)
    lo, hi = bootstrap_auc_ci(yte, pt) if name != 'baseline_prior' else (0.5, 0.5)
    rows.append({'model': name, 'precision': r['precision'], 'recall': r['recall'], 'roc_auc': r['roc_auc'],
                 'auc_95ci': f'{lo:.3f} to {hi:.3f}', 'threshold': r['threshold'], 'val_auc': m.get('val_auc', np.nan)})
res = pd.DataFrame(rows).set_index('model')
res.round(3)

,precision,recall,roc_auc,auc_95ci,threshold,val_auc
model,,,,,,
baseline_prior,0.309,1.000,0.500,0.500 to 0.500,0.05,NaN
baseline_minutes,0.398,0.816,0.706,0.646 to 0.764,0.37,NaN
logistic_regression,0.469,0.670,0.737,0.679 to 0.794,0.34,0.761
gradient_boosting,0.490,0.718,0.745,0.686 to 0.798,0.29,0.737
pytorch_mlp,0.462,0.757,0.740,0.682 to 0.796,0.33,0.764


In [6]:
print(f'Test rows {len(test)}, regular starters {int(yte.sum())} ({yte.mean():.1%}). Run date {RUN_DATE}.')
import os
os.makedirs('../results', exist_ok=True)
res.round(4).to_csv('../results/step2_model_results.csv')
funnel.to_csv('../results/step2_record_funnel.csv', index=False)
md = ['# Step 2 results', '', f'Run date: {RUN_DATE}. Snapshot: Transfermarkt 2026-07-06.', '',
      f'Pre transfer records: {len(df)} (train {len(train)}, validation {len(val)}, test {len(test)}, embargo {len(embargo)} not used).',
      f'Test regular starter rate: {yte.mean():.1%}.', '',
      '| model | precision | recall | ROC AUC | AUC 95% CI (bootstrap) |', '|---|---|---|---|---|']
for k, r in res.iterrows():
    md.append(f"| {k} | {r.precision:.3f} | {r.recall:.3f} | {r.roc_auc:.3f} | {r.auc_95ci} |")
md += ['', 'Selected hyperparameters (by validation ROC AUC):', '']
for k, v in models.items():
    md.append(f'- {k}: {json.dumps(v["params"])}')
open('../results/step2_results.md', 'w').write('\n'.join(md) + '\n')
print('\n'.join(md))

Test rows 333, regular starters 103 (30.9%). Run date 2026-10-15.
# Step 2 results

Run date: 2026-10-15. Snapshot: Transfermarkt 2026-07-06.

Pre transfer records: 1669 (train 694, validation 246, test 333, embargo 396 not used).
Test regular starter rate: 30.9%.

| model | precision | recall | ROC AUC | AUC 95% CI (bootstrap) |
|---|---|---|---|---|
| baseline_prior | 0.309 | 1.000 | 0.500 | 0.500 to 0.500 |
| baseline_minutes | 0.398 | 0.816 | 0.706 | 0.646 to 0.764 |
| logistic_regression | 0.469 | 0.670 | 0.737 | 0.679 to 0.794 |
| gradient_boosting | 0.490 | 0.718 | 0.745 | 0.686 to 0.798 |
| pytorch_mlp | 0.462 | 0.757 | 0.740 | 0.682 to 0.796 |

Selected hyperparameters (by validation ROC AUC):

- baseline_prior: {"prior": 0.326}
- baseline_minutes: {"rule": "minutes_365 / 3400"}
- logistic_regression: {"C": 0.01}
- gradient_boosting: {"learning_rate": 0.03, "max_depth": 2, "n_estimators": 100}
- pytorch_mlp: {"hidden": [32, 16], "dropout": 0.4, "weight_decay": 0.01, "epochs_ru

## 4. Reading the table

Precision and recall use a single threshold chosen on the validation set, so they depend on that choice. ROC AUC does not. With 333 test moves the AUC interval is wide: differences of a few hundredths between the learned models are not meaningful. The honest reading is in the markdown file written above.

## 5. Does the shortlist rule rank better than chance? (backtest on test moves)

In [7]:
best_name = max([k for k in models if 'val_auc' in models[k]], key=lambda k: models[k]['val_auc'])
print('model chosen by validation AUC:', best_name)
t = test.copy()
t['p'] = models[best_name]['predict'](t)
t['year'] = t.transfer_date.dt.year
t = t[t.value_before.notna()].copy()
t['p_pct'] = t.groupby('from_league').p.rank(pct=True)
t['v_pct'] = t.groupby('from_league').value_before.rank(pct=True)
t['score'] = t.p_pct - t.v_pct
def lift(col):
    cut = t[col].quantile(0.8)
    top = t[t[col] >= cut]
    return len(top), top.regular_starter.mean()
base = t.regular_starter.mean()
out = pd.DataFrame({'moves': [len(t)] + [lift(c)[0] for c in ['p', 'score']],
                    'starter_rate': [base] + [lift(c)[1] for c in ['p', 'score']]},
                   index=['all test moves', 'top 20% by predicted probability', 'top 20% by shortlist score (prob rank minus value rank)'])
out.round(3)

model chosen by validation AUC: pytorch_mlp


,moves,starter_rate
all test moves,333,0.309
top 20% by predicted probability,67,0.567
top 20% by shortlist score (prob rank minus value rank),67,0.493


In [8]:
out.round(4).to_csv('../results/step2_shortlist_backtest.csv')